In [ ]:
# Notebook: Interactive Band-Stop Filter Analysis (Ideal & Real)
# Goal: Dynamically explore band-stop filter frequency responses with two cutoff frequencies \omega_{c_1} and \omega_{c_2}.

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def plot_interactive_band_stop(omega_c1, omega_c2):
    # Ensure omega_c2 is strictly greater than omega_c1
    if omega_c2 <= omega_c1:
        omega_c2 = omega_c1 + 1.0

    omega_vals = np.linspace(-25, 25, 1200)
    
    # 1. Ideal Band-Stop Magnitude Response: 0 for \omega_{c_1} <= |\omega| <= \omega_{c_2}, else 1
    ideal_bs = np.where((np.abs(omega_vals) >= omega_c1) & (np.abs(omega_vals) <= omega_c2), 0.0, 1.0)
    
    # 2. Real Band-Stop Magnitude Response with smooth transition bands and ripples
    real_bs = np.zeros_like(omega_vals)
    trans_width = 2.0
    
    for i, w in enumerate(np.abs(omega_vals)):
        if w < omega_c1 - trans_width:
            # Lower passband
            real_bs[i] = 1.0 + 0.04 * np.cos(3 * w)
        elif omega_c1 - trans_width <= w < omega_c1:
            # Lower transition band falling from 1.0 to 0.7071 and down
            fraction = (w - (omega_c1 - trans_width)) / trans_width
            real_bs[i] = 0.5 * (1.0 + np.cos(np.pi * fraction))
        elif omega_c1 <= w <= omega_c2:
            # Stopband with attenuation ripples
            real_bs[i] = 0.04 * np.sin(2 * w)
        elif omega_c2 < w <= omega_c2 + trans_width:
            # Upper transition band rising from 0 to 0.7071 and up to 1.0
            fraction = (w - omega_c2) / trans_width
            real_bs[i] = 0.5 * (1.0 - np.cos(np.pi * fraction))
        else:
            # Upper passband
            real_bs[i] = 1.0 + 0.04 * np.cos(3 * w)

    # Create figure with two subplots: Ideal BS vs Real BS
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
    
    # Plot Ideal Band-Stop Filter
    ax1.plot(omega_vals, ideal_bs, 'b-', linewidth=2.5, label=r"Ideal BS $|\mathcal{H}_{bs}(j\omega)|$")
    ax1.axvline(omega_c1, color='red', linestyle='--', linewidth=1.5, label=r"Cutoffs $\omega_{c_1}=%.1f, \omega_{c_2}=%.1f$" % (omega_c1, omega_c2))
    ax1.axvline(omega_c2, color='red', linestyle='--', linewidth=1.5)
    ax1.axvline(-omega_c1, color='red', linestyle='--', linewidth=1.5)
    ax1.axvline(-omega_c2, color='red', linestyle='--', linewidth=1.5)
    ax1.set_title(r"Interactive Ideal and Real Band-Stop Filter Responses", fontsize=13)
    ax1.set_ylabel(r"Amplitude $|\mathcal{H}(j\omega)|$", fontsize=11)
    ax1.grid(True, linestyle=":", alpha=0.7)
    ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax1.set_ylim(-0.15, 1.25)
    ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=10)
    
    # Plot Real Band-Stop Filter
    ax2.plot(omega_vals, real_bs, 'g-', linewidth=2, label=r"Real BS (ripples & transitions)")
    ax2.axhline(0.7071, color='orange', linestyle=':', linewidth=1.5, label=r"-3 dB point ($0.7071$)")
    ax2.axvline(omega_c1, color='red', linestyle='--', linewidth=1.5, label=r"Cutoffs $\omega_{c_1}, \omega_{c_2}$")
    ax2.axvline(omega_c2, color='red', linestyle='--', linewidth=1.5)
    ax2.axvline(-omega_c1, color='red', linestyle='--', linewidth=1.5)
    ax2.axvline(-omega_c2, color='red', linestyle='--', linewidth=1.5)
    ax2.set_xlabel(r"Frequency $\omega$ (rad/s)", fontsize=11)
    ax2.set_ylabel(r"Amplitude $|\mathcal{H}(j\omega)|$", fontsize=11)
    ax2.grid(True, linestyle=":", alpha=0.7)
    ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax2.set_ylim(-0.15, 1.25)
    ax2.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=10)
    
    plt.tight_layout()
    plt.show()

# Interactive widget sliders for two cutoff frequencies
widgets.interact(
    plot_interactive_band_stop, 
    omega_c1=widgets.FloatSlider(value=3.0, min=1.0, max=10.0, step=0.5, description=r'$\omega_{c_1}$:', style={'description_width': 'initial'}),
    omega_c2=widgets.FloatSlider(value=9.0, min=4.0, max=18.0, step=0.5, description=r'$\omega_{c_2}$:', style={'description_width': 'initial'})
);